In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
train["text"] = (
    "Question: " + train["prompt"].fillna("") +
    " A: " + train["A"].fillna("") +
    " B: " + train["B"].fillna("") +
    " C: " + train["C"].fillna("") +
    " D: " + train["D"].fillna("") +
    " E: " + train["E"].fillna("")
)

test["text"] = (
    "Question: " + test["prompt"].fillna("") +
    " A: " + test["A"].fillna("") +
    " B: " + test["B"].fillna("") +
    " C: " + test["C"].fillna("") +
    " D: " + test["D"].fillna("") +
    " E: " + test["E"].fillna("")
)

In [ ]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

In [ ]:
from transformers import AutoTokenizer

MODEL = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
import wandb

wandb.login(key="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")

wandb.init(
    project="24f3004935-t22026"
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train[["text","label"]]
)

test_ds = Dataset.from_pandas(
    test[["text"]]
)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=5
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="wandb"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)

trainer.train()

In [ ]:
import numpy as np
import torch

pred = trainer.predict(test_ds)

probs = torch.softmax(
    torch.tensor(pred.predictions),
    dim=1
).numpy()

predictions = []

for p in probs:
    top3 = np.argsort(p)[::-1][:3]
    predictions.append(
        " ".join([id2label[i] for i in top3])
    )

# MILESTONE 1

In [ ]:
print(train['answer'].value_counts(ascending=False))
freq_dist = list(train['answer'].value_counts(ascending=False))
print(freq_dist[0]+freq_dist[-1])

In [ ]:
import string

vocab = set()

for text in train["prompt"]:
    # lowercase
    text = str(text).lower()

    # remove standard punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # split by whitespace
    words = text.split()

    vocab.update(words)

print("Vocabulary Size:", len(vocab))

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# Row ID 1
text = train.loc[train["id"] == 1, "prompt"].iloc[0]

# lowercase + remove punctuation
text = text.lower()
text = text.translate(str.maketrans('', '', string.punctuation))

# tokenize
words = text.split()

# remove stopwords
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print(filtered_words)
print("Count:", len(filtered_words))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)

print("Vocabulary Size:", len(vectorizer.get_feature_names_out()))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

row = train.iloc[0]   # Row ID 1 if first row

prompt_vec = vectorizer.transform([row["prompt"]])
A_vec = vectorizer.transform([row["A"]])

score = cosine_similarity(prompt_vec, A_vec)[0][0]

print(round(score, 4))

In [ ]:
correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        scores[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    predicted = max(scores, key=scores.get) # highest cosine similarity

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / len(train)) * 100

print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
def map_at_3(actual, predicted):
    try:
        rank = predicted.index(actual) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0

actual = "C"
predicted = ["C", "A", "B"]

score = map_at_3(actual, predicted)

print(score)

In [ ]:
actual = "B"
predicted = ["D", "B", "E"]

score = map_at_3(actual, predicted)

print(score)

In [ ]:
top3 = train["answer"].value_counts().index[:3].tolist()

prediction = top3

score = 0

for ans in train["answer"]:

    if ans == prediction[0]:
        score += 1.0

    elif ans == prediction[1]:
        score += 0.5

    elif ans == prediction[2]:
        score += 1/3

map3 = score / len(train)

print("Top 3:", prediction)
print("MAP@3:", round(map3, 4))

# MILESTONE 2

## Preprocessing

In [ ]:
from datasets import load_dataset

# Load train.csv
Train = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# Create combined_text = prompt + " " + A
Train = Train.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

# Character length of row 51
print(len(Train[51]["combined_text"]))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

In [ ]:
print(Train)
print(type(Train))

In [ ]:
print(type(Train["prompt"]))
print(type(Train["prompt"][0]))
print(Train["prompt"][0])

In [ ]:
encodings = tokenizer(
    list(Train["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

hidden_size = config.hidden_size
num_heads = config.num_attention_heads

head_dim = hidden_size // num_heads

print("Hidden Size:", hidden_size)
print("Attention Heads:", num_heads)
print("Head Dimension:", head_dim)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Tokenize row 0 prompt (default settings)
inputs = tokenizer(Train[0]["prompt"], return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Print shape
print(outputs.last_hidden_state.shape)

In [ ]:
# Extract CLS embedding (batch 0, token 0)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum first 5 values
answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

In [ ]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

inputs = tokenizer("Light-ion fusion is a technique.", return_tensors="pt")
outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1][0, 0]

print(round(attention[0, fusion_index].item(), 4))

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B from row 0
prompt = Train[0]["prompt"]
option_b = Train[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(round(similarity.item(), 4))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mini_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

options = ["A", "B", "C", "D", "E"]

mini_map = 0
improved = 0

for row in Train:

    prompt = row["prompt"]
    answer = row["answer"]

    # Pipeline 1 : TF-IDF

    docs = [prompt] + [row[o] for o in options]

    tfidf = TfidfVectorizer()
    X = tfidf.fit_transform(docs)

    sims = cosine_similarity(X[0], X[1:]).flatten()

    tfidf_rank = [
        options[i]
        for i in sims.argsort()[::-1]
    ]


    # Pipeline 2 : MiniLM

    prompt_emb = mini_model.encode(prompt, convert_to_tensor=True)
    option_embs = mini_model.encode(
        [row[o] for o in options],
        convert_to_tensor=True
    )

    sims = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()

    mini_rank = [
        options[i]
        for i in sims.argsort()[::-1]
    ]

    # MAP@3
    top3 = mini_rank[:3]

    if answer in top3:
        mini_map += 1 / (top3.index(answer) + 1)


    # Improvement count
    tfidf_hit = answer in tfidf_rank[:3]
    mini_hit = answer in mini_rank[:3]

    if (not tfidf_hit) and mini_hit:
        improved += 1

mini_map /= len(Train)

print("MiniLM MAP@3:", round(mini_map, 4))
print("Improved count:", improved)

In [ ]:
from transformers import pipeline
# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
prompt = Train[1]["prompt"]

# Candidate labels: Options A, B, C
candidate_labels = [
    Train[1]["A"],
    Train[1]["B"],
    Train[1]["C"]
]

# Predict
result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# Probability score of the top-ranked option
print(round(result["scores"][0], 4))

In [ ]:
result_softmax = classifier(
    prompt,
    candidate_labels=candidate_labels
)

# This question (Independent Sigmoids)
result_sigmoid = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

print("Softmax sum:", softmax_sum)
print("Sigmoid sum:", sigmoid_sum)
print("Absolute difference:", round(abs(softmax_sum - sigmoid_sum), 4))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

text = (
    f"Question: {Train[0]['prompt']}. "
    f"Is the correct answer A: {Train[0]['A']} "
    f"or B: {Train[0]['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# SUBMISSION

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()